<a href="https://colab.research.google.com/github/ninjafox250/yolo-model-training/blob/main/train_yolo_modelsV2-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics scikit-learn iterative-stratification


In [ ]:
# Unzip images to a custom data folder
!unzip -q /content/dataset.zip -d /content/

In [ ]:
# =====================
# IMPORTS
# =====================
import shutil
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# =====================
# PATHS (COLAB)
# =====================
SRC_IMG_DIR = Path("/content/dataset/images")
SRC_LBL_DIR = Path("/content/dataset/labels")

BASE_OUT = Path("/content/data")

TRAIN_IMG_DIR = BASE_OUT / "train/images"
TRAIN_LBL_DIR = BASE_OUT / "train/labels"
VAL_IMG_DIR = BASE_OUT / "validation/images"
VAL_LBL_DIR = BASE_OUT / "validation/labels"
HOLDOUT_DIR = BASE_OUT / "holdout"

IMG_EXTS = [".jpg", ".jpeg", ".png"]

# =====================
# CLEAN (optional but recommended in notebooks)
# =====================
shutil.rmtree(BASE_OUT, ignore_errors=True)

# =====================
# CREATE DIRS
# =====================
for d in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, VAL_IMG_DIR, VAL_LBL_DIR, HOLDOUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================
# LOAD DATA (image-first)
# =====================
image_files = []
labels_per_image = []

for img_path in SRC_IMG_DIR.iterdir():
    if img_path.suffix.lower() not in IMG_EXTS:
        continue

    lbl_path = SRC_LBL_DIR / f"{img_path.stem}.txt"

    if not lbl_path.exists():
        continue

    classes = set()
    with open(lbl_path) as f:
        for line in f:
            if line.strip():
                classes.add(int(line.split()[0]))

    image_files.append(img_path)
    labels_per_image.append(list(classes))

print(f"Total usable dataset: {len(image_files)}")

# =====================
# ENCODE LABELS
# =====================
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(labels_per_image)

# =====================
# STEP 1: HOLDOUT (10%)
# =====================
holdout_split = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

trainval_idx, holdout_idx = next(holdout_split.split(image_files, Y))

trainval_images = [image_files[i] for i in trainval_idx]
trainval_labels = [labels_per_image[i] for i in trainval_idx]

holdout_images = [image_files[i] for i in holdout_idx]

print(f"Holdout: {len(holdout_images)}")
print(f"Train/Val pool: {len(trainval_images)}")

# =====================
# STEP 2: TRAIN/VAL (80/20)
# =====================
mlb2 = MultiLabelBinarizer()
Y_trainval = mlb2.fit_transform(trainval_labels)

train_split = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(train_split.split(trainval_images, Y_trainval))

train_images = [trainval_images[i] for i in train_idx]
val_images = [trainval_images[i] for i in val_idx]

print(f"Train: {len(train_images)}")
print(f"Validation: {len(val_images)}")

# =====================
# COPY FUNCTIONS
# =====================
def copy_yolo_pairs(image_list, img_dst, lbl_dst):
    for img_path in image_list:
        lbl_path = SRC_LBL_DIR / f"{img_path.stem}.txt"

        shutil.copy2(img_path, img_dst / img_path.name)
        shutil.copy2(lbl_path, lbl_dst / lbl_path.name)

def copy_holdout(image_list, dst):
    for img_path in image_list:
        shutil.copy2(img_path, dst / img_path.name)

# =====================
# EXECUTE COPY
# =====================
copy_yolo_pairs(train_images, TRAIN_IMG_DIR, TRAIN_LBL_DIR)
copy_yolo_pairs(val_images, VAL_IMG_DIR, VAL_LBL_DIR)
copy_holdout(holdout_images, HOLDOUT_DIR)

print("✅ Done.")

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/dataset/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640

In [ ]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=/content/data/holdout save=True

In [ ]:
from datetime import datetime

now = datetime.now()
modelname = now.strftime("%b").lower() + f"-{now.day}-{now.strftime('%y')}_{now.hour}"

!mkdir -p /content/$modelname

!cp /content/runs/detect/train/weights/best.pt /content/$modelname/$modelname.pt
!cp -r /content/runs/detect/predict /content/$modelname/
!cp -r /content/data/holdout /content/$modelname/

!cd /content && zip -r $modelname.zip $modelname